# Easily generate an update script

## Load useful libraries

In [ ]:
import os

## Declare configuration settings

In [ ]:
output_file_commands = '/tmp/update_script.sh'
output_file_crontab = '/tmp/crontab_commands.txt'

instrument_list = [
    'AUD/USD',
    'EUR/USD',
    'GBP/USD',
    'NZD/USD',
    'USD/CAD',
    'USD/CHF',
    'USD/JPY',
]

granularity_list = ['D', 'H1', 'M15']
granularity_list_ff = ['H1', 'M15']

cron_strings_map = {
    'D' : '0 0 * * *',
    'H1' : '0 * * * *',
    'M15' : '*/15 * * * *',
}

## Retrieve required environment variables

In [ ]:
home = os.environ['HOME']
forex_home = os.environ['FOREX_HOME']
python_tools_and_shortcuts_home = os.environ['PYTHON_TOOLS_AND_SHORTCUTS_HOME']
creds_file = os.environ['CREDS']

## Iterate through the instruments/granularities

In [ ]:
cmd_list = []
for instrument in instrument_list:
    instrument_file_name = instrument.replace('/', '_')
    for granularity in granularity_list:
        
        cmd = f'''
            export HOME={home};
            export FOREX_HOME={forex_home};
            export PYTHON_TOOLS_AND_SHORTCUTS_HOME={python_tools_and_shortcuts_home};
            export PYTHONPATH=$PYTHONPATH:$PYTHON_TOOLS_AND_SHORTCUTS_HOME:$FOREX_HOME;
            export CREDS={creds_file};
            source $HOME/venvs/core/bin/activate;
            $HOME/venvs/core/bin/papermill $FOREX_HOME/forex/pipelines/pipeline-outline.ipynb
            /tmp/pipeline-outline_{instrument_file_name}_{granularity}.ipynb
            -p granularity {granularity}
            -p instrument "{instrument}"
            -p config_file $CREDS'''

        cmd = ' '.join([x.strip() for x in cmd.split('\n') if x.strip() != ''])
        cmd_list.append(cmd)

## Write commands and crontab commands to text files

In [ ]:
with open(output_file_commands, 'w') as f:
    for cmd in cmd_list:
        f.write(cmd + '\n\n')

In [ ]:
with open(output_file_crontab, 'w') as f:
    for cmd in cmd_list:
        granularity = cmd.split('granularity')[-1].split('-p')[0].strip()
        f.write(cron_strings_map[granularity] + ' ' + cmd + '\n\n')